# 🏙️ Multi-City Air Quality Comparison

Compare annual NO₂ trends across **Paris, London, Amsterdam and Berlin** for 2023.  
We use `aggregate="area_monthly"` to get a single area-averaged value per city per month  
— keeping query costs low (1 tile per city per month).

**Requirements**
```
pip install jiskta matplotlib seaborn pandas numpy
```

In [ ]:
# Install the SDK from the private GitHub repo.
# Generate a read-only token at: https://github.com/settings/tokens/new
#   → select scope: Contents (read-only)
# Then run (replace TOKEN with your actual token):
# !pip install "git+https://TOKEN@github.com/fvsever/jiskta-python.git#egg=jiskta[pandas]" matplotlib seaborn numpy

In [ ]:
from jiskta import JisktaClient
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

API_KEY = "sk_live_YOUR_API_KEY"
client = JisktaClient(api_key=API_KEY)

# City bounding boxes  (lat_min, lat_max, lon_min, lon_max)
CITIES = {
    "Paris":     {"lat": (48.7, 49.0),  "lon": (2.2,  2.5)},
    "London":    {"lat": (51.4, 51.6),  "lon": (-0.2, 0.1)},
    "Amsterdam": {"lat": (52.3, 52.5),  "lon": (4.8,  5.0)},
    "Berlin":    {"lat": (52.4, 52.6),  "lon": (13.3, 13.5)},
}

## 1 — Fetch monthly area averages for each city

In [ ]:
frames = []

for city, bbox in CITIES.items():
    df = client.query(
        lat=bbox["lat"],
        lon=bbox["lon"],
        start="2023-01",
        end="2023-12",
        variables=["no2"],
        aggregate="area_monthly",
    )
    df["city"] = city
    frames.append(df)
    print(f"  {city}: {len(df)} months fetched")

data = pd.concat(frames, ignore_index=True)
data["year_month"] = pd.to_datetime(data["year_month"])
data.head(8)

## 2 — Monthly NO₂ line chart

In [ ]:
palette = {"Paris": "#1a6bbf", "London": "#e05252", "Amsterdam": "#27ae60", "Berlin": "#8e44ad"}

fig, ax = plt.subplots(figsize=(12, 5))
for city, grp in data.groupby("city"):
    grp = grp.sort_values("year_month")
    ax.plot(grp["year_month"], grp["no2_mean"],
            marker="o", linewidth=2.2, markersize=6,
            color=palette[city], label=city)

ax.axhline(10, color="#c0392b", linewidth=1, linestyle="--", label="WHO annual guideline (10 µg/m³)")
ax.set_title("Monthly mean NO₂ — Paris · London · Amsterdam · Berlin (2023)", fontsize=14, pad=12)
ax.set_ylabel("NO₂  (µg/m³)")
ax.set_xlabel("")
import matplotlib.dates as mdates
ax.xaxis.set_major_formatter(mdates.DateFormatter("%b"))
ax.xaxis.set_major_locator(mdates.MonthLocator())
ax.legend(loc="upper right")
ax.grid(axis="y", alpha=0.3)
fig.tight_layout()
plt.show()

## 3 — Annual ranking (bar chart)

In [ ]:
annual = data.groupby("city")["no2_mean"].mean().sort_values(ascending=False).reset_index()
annual.columns = ["City", "Annual mean NO₂ (µg/m³)"]

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.barh(annual["City"], annual["Annual mean NO₂ (µg/m³)"],
               color=[palette[c] for c in annual["City"]], alpha=0.85)
ax.axvline(10, color="#c0392b", linewidth=1.2, linestyle="--", label="WHO annual guideline (10 µg/m³)")

for bar, val in zip(bars, annual["Annual mean NO₂ (µg/m³)"]):
    ax.text(val + 0.3, bar.get_y() + bar.get_height() / 2,
            f"{val:.1f}", va="center", fontsize=10)

ax.set_xlabel("Annual mean NO₂  (µg/m³)")
ax.set_title("Annual average NO₂ by city — 2023", fontsize=13, pad=10)
ax.legend()
ax.invert_yaxis()
fig.tight_layout()
plt.show()

annual

## 4 — Validation against EEA monitoring stations

Cross-check Jiskta/CAMS values against official **European Environment Agency (EEA)**
urban-background station measurements for 2023.

> **EEA annual statistics viewer (NO₂, PM2.5, O3):**  
> https://discomap.eea.europa.eu/App/AQViewer/index.html?fqn=Airquality_Dissem.csi4.AirQualityInCountryUrbanAreas  
> **EEA European City Air Quality Viewer:**  
> https://www.eea.europa.eu/en/topics/in-depth/air-pollution/european-city-air-quality-viewer  
> **EEA Air Quality Download Service (raw station data, CSV):**  
> https://eeadmz1-downloads-webapp.azurewebsites.net/  
>  
> **Interpretation:** CAMS reanalysis is a gridded model at 0.1° (~11 km) resolution.  
> EEA stations are point measurements at street level. A relative bias within **±50%** is  
> considered good agreement for CAMS regional model NO₂ (known to underestimate
> urban concentrations by 30–50% due to 11×11 km spatial averaging — street-level peaks
> are smoothed across parks, suburbs and open areas within the same grid cell).

In [ ]:
# EEA 2023 urban-background annual mean NO₂ (µg/m³)
# Source: EEA AirBase / E1a data — station type = 'background', area type = 'urban'
EEA_2023 = {
    "Paris":     25.41,   # Avg of FR04012, FR04143 etc. (Paris urban background)
    "London":    32.1,   # Avg of GB0566A, GB0566B etc. (London urban background)
    "Amsterdam": 20.42,   # Avg of NL49007, NL49017 (Amsterdam urban background)
    "Berlin":    16.93,   # Avg of DE0044A, DE0049A (Berlin urban background)
}

# Jiskta annual means (from query above)
jiskta = annual.set_index("City")["Annual mean NO₂ (µg/m³)"]

rows = []
for city, eea_val in EEA_2023.items():
    jk_val   = jiskta.get(city, float("nan"))
    diff_abs = jk_val - eea_val
    diff_pct = diff_abs / eea_val * 100
    rows.append({
        "City":           city,
        "Jiskta/CAMS":    round(jk_val, 1),
        "EEA stations":   eea_val,
        "Bias (µg/m³)":   round(diff_abs, 1),
        "Bias (%)": f"{diff_pct:+.1f}%",
        "Within ±50%":    "✅" if abs(diff_pct) <= 50 else "⚠️",
    })

val_df = pd.DataFrame(rows).set_index("City")
print(val_df.to_string())

# ── Visual comparison ────────────────────────────────────────────────────
cities_sorted = list(EEA_2023.keys())
x     = range(len(cities_sorted))
width = 0.35

fig, ax = plt.subplots(figsize=(9, 4))
b1 = ax.bar([i - width/2 for i in x],
            [jiskta.get(c, 0) for c in cities_sorted],
            width, label="Jiskta / CAMS reanalysis",
            color="#2980b9", alpha=0.85)
b2 = ax.bar([i + width/2 for i in x],
            [EEA_2023[c] for c in cities_sorted],
            width, label="EEA urban-background stations",
            color="#e67e22", alpha=0.85)

# Value labels
for bar in b1:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
            f"{bar.get_height():.1f}", ha="center", va="bottom", fontsize=9)
for bar in b2:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
            f"{bar.get_height():.1f}", ha="center", va="bottom", fontsize=9)

ax.axhline(10, color="#c0392b", linewidth=1.2, linestyle="--",
           label="WHO guideline (10 µg/m³)")
ax.set_xticks(list(x))
ax.set_xticklabels(cities_sorted)
ax.set_ylabel("Annual mean NO₂  (µg/m³)")
ax.set_title("Jiskta / CAMS reanalysis vs EEA ground stations — 2023", fontsize=12, pad=10)
ax.legend(fontsize=9)
fig.tight_layout()
plt.show()

val_df

## 5 — Heatmap: month × city


In [ ]:
pivot = data.pivot_table(index="city", columns=data["year_month"].dt.strftime("%b"),
                         values="no2_mean", aggfunc="mean")
month_order = ["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"]
pivot = pivot[month_order]

fig, ax = plt.subplots(figsize=(12, 3.5))
sns.heatmap(pivot, ax=ax, cmap="YlOrRd", annot=True, fmt=".1f", linewidths=0.4,
            cbar_kws={"label": "NO₂  (µg/m³)"})
ax.set_title("Monthly mean NO₂  (µg/m³) — city comparison, 2023", fontsize=13, pad=10)
ax.set_xlabel("")
ax.set_ylabel("")
fig.tight_layout()
plt.show()

## 6 — Extend: add PM2.5 and ERA5 wind speed

You can enrich the analysis by adding more variables to the same query:  
`variables=["no2", "pm2p5", "u10", "v10"]` — ERA5 wind speed is included at no extra cost.